In [ ]:
from google import genai
import sys
import pandas as pd
import ace_lib as ace
import helpful_functions as hf
import nest_asyncio
import asyncio
from time import sleep
import nest_asyncio
nest_asyncio.apply()

In [ ]:
brain_session= ace.start_session()

Complete biometrics authentication and press any key to continue: 
https://api.worldquantbrain.com/authentication/persona?inquiry=inq_AUAgjsUhtuQNkVg9f4Q3YkGxU1idNc



In [ ]:
rg= 'GLB'
un='MINVOL1M'
dataset_ids=["pv1", "model110", "model77", "model26", "model109", "risk70", "model38", "model50", "analyst69", "model264", "other460", "fundamental23", "other455", "model138" ]

In [ ]:
import os
import json

# Schema for LLM-generated alphas
alpha_generation_schema = {
    "type": "object",
    "properties": {
        "alphas": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string"},
                    "rationale": {"type": "string"}
                },
                "required": ["expression", "rationale"]
            }
        },
        "summary": {"type": "string"}
    },
    "required": ["alphas"]
}


async def call_llm(prompt, json_schema=None, model="gemini-2.5-flash", api_key=None, system_message=None, max_retries=10, mock_response=None):
    """Call GenAI chat completion and return parsed JSON when schema supplied.

    Parameters:
    - prompt: str
    - json_schema: dict or None
    - model: model name
    - api_key: optional override for the GenAI client
    - system_message: optional system message string
    - max_retries: number of retries on transient errors
    - mock_response: if set, returned as-is (useful for tests)
    """
    api_key = api_key or os.environ.get("GENAI_API_KEY")
    if not api_key:
        raise RuntimeError("No GenAI API key provided; set GENAI_API_KEY or pass api_key")

    """client = genai.Client(
    vertexai=True,
    project="project-986d6f59-924d-4245-9c3",
    location="us-central1"
) """
    client = genai.Client(
        api_key=api_key)
    system_msg = system_message or "You are a quantitative finance expert. When a schema is supplied, respond only in JSON matching the schema."

    last_exc = None
    for attempt in range(1, max_retries + 1):
        try:
            response = client.models.generate_content(
                model=model,
                contents=prompt,
                config={
        "response_mime_type": "application/json",
        "response_json_schema": json_schema,
    },            )

            content = response.text
            if isinstance(content, dict):
                return content
            try:
                return json.loads(content.strip())
            except Exception:
                return content.strip()
        except Exception as e:
            last_exc = e
            print(f"LLM call error (attempt {attempt}/{max_retries}): {e}")
            if attempt < max_retries:
                await asyncio.sleep(1 * attempt)
                continue
            raise

    # If we exhausted retries
    raise last_exc

In [ ]:
# Clean helper functions for prompt building and JSON-driven generation

def build_initial_prompt(num_alphas, operators, data_fields):
    txt = (
        "Return a JSON object matching the schema with key 'alphas' (each item should contain 'expression' and 'rationale') and an optional 'summary'.",
        "Operators: {operators}",
        "Data: {data}",
        "DONT USE DATAFIELDS OUTSIDE THE ONE GIVEN UNDER ANY CONDITION"
        "Constraints:",
        "  - type=MATRIX fields can be used directly as input to Arithmetic, Cross Sectional, Time Series, Logical, and Transformational operators, as a group in Group operators, or with bucket().",
        "  - type=VECTOR fields CANNOT be used directly. Always wrap type=VECTOR data in [vec_count, vec_max, vec_sum, vec_avg, vec_stddev, vec_min, vec_range] operator first, then treat output as MATRIX.",
        "  - type=GROUP fields CANNOT be used directly. Use them only as 'group' parameter in Group operators.",
        "  - If a data field has low coverage, you may need to use `ts_backfill` to clean the field first(only works on MATRIX fields).",
        "  - DONT use datafields outside of the provided ones.",
        "  - Provide only JSON as per schema.",
        "  - Try to make single dataset alphas where possible.",
        f"Generate {num_alphas} new alpha expressions",
    )
    return '\n'.join(txt).format(n=num_alphas, operators=operators[operators['scope']=='REGULAR'].to_json(), data=data_fields.to_json())


def build_variation_prompt(prev_summary, prev_alphas, operators, data_fields, num_alphas, variation_instructions=None):
    """Build a variation prompt that includes only the top 2 previous alphas by sharpe.

    - If a previous alpha doesn't have a sharpe value, it will be excluded.
    - If no prior alphas have sharpe values, an empty list will be passed.
    """
    prev_brief = prev_summary if prev_summary else 'No prior summary available.'

    # Filter previous alphas to those that have a sharpe, sort by sharpe descending, and take top 2
    scored = []
    for a in prev_alphas or []:
        s = prev_alphas[a][0]
        t = prev_alphas[a][2]
        if s is not None and t<0.5:
            scored.append((a, [s,prev_alphas[a][1]]))
    scored.sort(key=lambda x: x[1][0], reverse=True)
    selected_alphas_pre = [[a,_] for a, _ in scored[:5]]

    prod=[]
    try:    
        for b in selected_alphas_pre or []:
            id = b[1][1]
            corr = ace.get_prod_corr(brain_session,id)
            if corr is not None:
                if b[1][0]>0:
                    p = abs(float(corr.iloc[0,4]))
                    prod.append((b[0],p))
                else:
                    p = float(corr.iloc[0,5])
        prod.sort(key=lambda x: x[1], reverse=False)
        selected_alphas = [a for a, _ in prod[:3]]
    except:
        selected_alphas = selected_alphas_pre
        pass


    if selected_alphas:
        print(f"Selected {len(selected_alphas)} previous alphas for variation prompt (top by sharpe).")
    else:
        print("No previous alphas with sharpe found; not including previous alphas in the variation prompt.")

    def _format_alpha(a):
        if isinstance(a, dict):
            expr = a
            sharpe = prev_alphas[a]
            parts = [f"Expression: {expr}"]
            if sharpe is not None:
                parts.append(f"PrevSharpe: {sharpe}")
            return ' | '.join(parts)
        return str(a)

    prev_alphas_str = '[' + ', '.join(_format_alpha(a) for a in selected_alphas) + ']'
    print("Previous alphas for variation prompt:", prev_alphas_str)

    return '\n'.join([
        "You are generating variations of existing alphas.",
        f"Previous generation summary: {prev_brief}",
        f"Previous alphas (expression, rationale, prev_sharpe, alpha_id if available): {prev_alphas_str}",
        f"Operators: {operators[operators['scope']=='REGULAR'].to_json()}",
        "DONT USE DATAFIELDS OUTSIDE THE ONE GIVEN UNDER ANY CONDITION"
        f"Data: {data_fields.to_json()}",
        "**Constraints:**",
        "  - type=MATRIX fields can be used directly as input to Arithmetic, Cross Sectional, Time Series, Logical, and Transformational operators, as a group in Group operators, or with bucket().",
        "  - type=VECTOR fields CANNOT be used directly. Always wrap type=VECTOR data in [vec_count, vec_max, vec_sum, vec_avg, vec_stddev, vec_min, vec_range] operator first, then treat output as MATRIX.",
        "  - type=GROUP fields CANNOT be used directly. Use them only as 'group' parameter in Group operators.",
        "  - If a data field has low coverage, you may need to use `ts_backfill` to clean the field first(only works on MATRIX fields).",
        "  - DONT use datafields outside of the provided ones.",
        "  - Make Complex Alpha expressions that combine multiple operators and low data fields.",
        "  - Provide only JSON as per schema.",
        "  - Try to make single dataset alphas where possible.",
        "  - Dont use pv1 fields without mixing them with other fields always mix pv1 fields.",
        f"Generate {num_alphas} new alpha expressions, each with a brief rationale. Return JSON matching the schema (alphas array + optional summary).",
    ])


async def generate_alphas_via_llm(prompt, num_alphas=8):
    resp = await call_llm(prompt, json_schema=alpha_generation_schema)
    if not resp:
        return [], 'LLM returned no response'
    alphas = resp.get('alphas', [])
    summary = resp.get('summary', '')
    return alphas, summary


async def generate_new_alphas_2nd(brain_session, num_alphas=8, data_fields_param=None):
    operators = ace.get_operators(brain_session)
    data = data_fields_param if data_fields_param is not None else globals().get('data_fields', None)
    prompt = build_initial_prompt(num_alphas, operators, data)
    alphas, summary = await generate_alphas_via_llm(prompt, num_alphas=num_alphas)
    return alphas, summary

In [ ]:
async def alphas_sim(alpha_expressions, brain_session):
    neutralizations = ['STATISTICAL','SUBINDUSTRY']
    print("Simulating expressions:", alpha_expressions)
    alpha_list = []
    for x in alpha_expressions:
        for nz in neutralizations:
            alpha_list.append(
                ace.generate_alpha(
                    x,
                    region=rg,
                    universe=un,
                    neutralization=nz,
                    max_trade="OFF",
                    delay=1,
                    visualization=False
                )
            )
    

    result = ace.simulate_alpha_list_multi(brain_session, alpha_list)
    alphas={}

    best_sharpe = None
    submitted = 0
    for r in result:
        if r.get('alpha_id'):
            submitted += 1
        try:
            s = r.get('is_stats', {}).get('sharpe')
            t = r.get('is_stats', {}).get('turnover')
            alphas[r.get('simulate_data', {}).get('regular')] =[s.max(),r.get('alpha_id'),t.max()]
            if s is not None:
                val = float(abs(s.max()) if hasattr(s, 'max') else (abs(s)))
                if best_sharpe is None or val > best_sharpe:
                    best_sharpe = val
        except Exception:
            pass

    summary = {'results': result, 'best_sharpe': best_sharpe, 'submitted_count': submitted}
    print(f"Simulation summary: {summary['best_sharpe']}")
    return summary, alphas




In [ ]:
async def iterate_alphas(initial_description, brain_session, iterations=3, num_alphas=10, data_fields_param=None):
    operators = ace.get_operators(brain_session)
    data = data_fields_param if data_fields_param is not None else globals().get('data_fields', None)
    if data is None:
        raise RuntimeError('data_fields is not available; pass it via data_fields_param or define `data_fields`.')
    prev_summary = initial_description
    prev_alphas = []
    history = []
    for it in range(iterations):
        print(f"Iteration {it+1}/{iterations}")
        if it == 0:
            generated, gen_summary = await generate_new_alphas_2nd(brain_session, num_alphas=num_alphas, data_fields_param=data)
        else:
            prompt = build_variation_prompt(prev_summary, prev_alphas, operators, data, num_alphas)
            generated, gen_summary = await generate_alphas_via_llm(prompt, num_alphas=num_alphas)

        expressions = [a['expression'] for a in generated]
        sim_summary,prev_alphas = await alphas_sim(expressions, brain_session)

        prev_summary = f"Gen summary: {gen_summary}. Best_sharpe:{sim_summary.get('best_sharpe')}, submitted:{sim_summary.get('submitted_count')}"
        history.append({'generated': generated, 'gen_summary': gen_summary, 'sim': sim_summary})
    return history



async def main():
    initial_description = 'Start with simple expressions using available data fields.'
    print('Starting iterative alpha generation')
    history = await iterate_alphas(initial_description, brain_session, iterations=2, num_alphas=30, data_fields_param=data_fields)
    for i, h in enumerate(history, 1):
        print(f"\n--- Iteration {i} ---")
        print('Generated:', [a['expression'] for a in h['generated']])
#        print('Gen summary:', h['gen_summary'])
#        print('Sim summary:', h['sim'])


In [ ]:

for i in range(0,100):
    try:
        data_fields = pd.concat(
            [ace.get_datafields(brain_session, region=rg, universe=un, dataset_id=dataset_id, data_type='ALL',delay=1) for dataset_id in dataset_ids],ignore_index=True
        )
    except Exception as e:
        print(f"Error fetching data fields (attempt {i+1}/10): {e}")   
        sleep(2)
    else:
        break
if data_fields is None:
    raise RuntimeError("Failed to fetch data fields after multiple attempts.")


Error fetching data fields (attempt 1/10): 'count'
Error fetching data fields (attempt 2/10): 'count'
Error fetching data fields (attempt 3/10): 'count'
Error fetching data fields (attempt 4/10): 'count'
Error fetching data fields (attempt 5/10): 'count'
Error fetching data fields (attempt 6/10): 'count'
Error fetching data fields (attempt 7/10): 'count'
Error fetching data fields (attempt 8/10): 'count'
Error fetching data fields (attempt 9/10): 'count'
Error fetching data fields (attempt 10/10): 'count'
Error fetching data fields (attempt 11/10): 'count'
Error fetching data fields (attempt 12/10): 'count'
Error fetching data fields (attempt 13/10): 'count'
Error fetching data fields (attempt 14/10): 'count'
Error fetching data fields (attempt 15/10): 'count'
Error fetching data fields (attempt 16/10): 'count'
Error fetching data fields (attempt 17/10): 'count'
Error fetching data fields (attempt 18/10): 'count'
Error fetching data fields (attempt 19/10): 'count'
Error fetching data f

In [ ]:
if 'alphaCount' in data_fields.columns:
    orig = len(data_fields)
    
    data_fields = data_fields[data_fields['alphaCount']<50].reset_index(drop=True)
    print(f"Filtered data_fields: {orig} -> {len(data_fields)}")
else:
    print("alphaCount column not found; no filtering applied")

Filtered data_fields: 8118 -> 6856


In [ ]:
"""if 'alphaCount' in data_fields.columns:
    orig = len(data_fields)
    
    data_fields = data_fields[data_fields['coverage']>0.3].reset_index(drop=True)
    print(f"Filtered data_fields: {orig} -> {len(data_fields)}")
else:
    print("alphaCount column not found; no filtering applied")"""

'if \'alphaCount\' in data_fields.columns:\n    orig = len(data_fields)\n\n    data_fields = data_fields[data_fields[\'coverage\']>0.3].reset_index(drop=True)\n    print(f"Filtered data_fields: {orig} -> {len(data_fields)}")\nelse:\n    print("alphaCount column not found; no filtering applied")'

In [ ]:
data_fields.drop(columns=['subcategory','category','region','delay','universe','pyramidMultiplier','themes','dataset_id','category_id','category_name','subcategory_id','dateCreated','userCount','coverage','dateCoverage'], inplace=True, errors='ignore')

In [ ]:
data_fields

,id,description,dataset,type,alphaCount,dataset_name,subcategory_name
0,isin,ISIN Value,"{'id': 'pv1', 'name': 'Price Volume Data for E...",SYMBOL,5,Price Volume Data for Equity,Price Volume
1,mdl177_5shortsentimentfactor_benchmark_fee,Implied loan rate or benchmark borrowing fee i...,"{'id': 'model77', 'name': 'Analysts' Factor Mo...",MATRIX,25,Analysts' Factor Model,Technical Models
2,mdl177_5shortsentimentfactor_days_to_cover,Days to Cover: number of days it would take fo...,"{'id': 'model77', 'name': 'Analysts' Factor Mo...",MATRIX,42,Analysts' Factor Model,Technical Models
3,mdl177_5shortsentimentfactor_dmd_conc,Demand value concentration: degree to which bo...,"{'id': 'model77', 'name': 'Analysts' Factor Mo...",MATRIX,33,Analysts' Factor Model,Technical Models
4,mdl177_5shortsentimentfactor_dmd_supply,"Demand-to-supply ratio in securities lending, ...","{'id': 'model77', 'name': 'Analysts' Factor Mo...",MATRIX,25,Analysts' Factor Model,Technical Models
...,...,...,...,...,...,...,...
6851,mdl138_vcer_4idp,"The PDI factor over 4 trailing quarters, measu...","{'id': 'model138', 'name': 'Stock Selection fr...",VECTOR,11,Stock Selection from Accounting-based Factors,ML/AI Models
6852,mdl138_vcer_5idp,Raw Performance Deviation Index over 5 trailin...,"{'id': 'model138', 'name': 'Stock Selection fr...",VECTOR,4,Stock Selection from Accounting-based Factors,ML/AI Models
6853,mdl138_vd_3idp,Performance Deviation Index over 3 quarters as...,"{'id': 'model138', 'name': 'Stock Selection fr...",VECTOR,3,Stock Selection from Accounting-based Factors,ML/AI Models
6854,mdl138_vd_4idp,Performance Deviation Index over 4 trailing qu...,"{'id': 'model138', 'name': 'Stock Selection fr...",VECTOR,11,Stock Selection from Accounting-based Factors,ML/AI Models


In [ ]:
data_fields = data_fields.sample(n=1000).reset_index(drop=True)

In [ ]:
for i in range(0,10):   
   for i in range(0,10):
      try:
         await main()
      except Exception as e:
         print(f"Error on runtime (attempt {i+1}/10): {e}")   
         sleep(20)
      else:
         break

Starting iterative alpha generation
Iteration 1/2
Simulating expressions: ['ts_delta(price_to_smartest_ebitda_ratio_year, 5)', 'normalize(analyst_ebitda_revision_ntm_3m)', 'inverse(add(forward_pe_ratio_mean_year2, 0.0001))', 'ts_std_dev(forecasted_surprise_pct_ebitda_12m, 20)', 'rank(vec_avg(mdl138_longsimeq_qpdi5_debt_equity))', 'ts_delay(analyst_recommendation_diffusion_1m, 5)', 'if_else(greater(price_volatility_score_2, 0.5), return_distribution_skewness, 0)', 'rank(market_implied_10yr_eps_growth_rate)', 'subtract(mdl264_sncc_class, ts_delay(mdl264_sncc_class, 1))', 'signed_power(inverse(add(count_analysts_lower_curr_qtr_earnings_30d, 0.0001)), 0.5)', 'zscore(lease_assets_present_balance)', 'group_zscore(vec_avg(analyst69_net_expected_report_period), oth455_customer_n2v_p10_q50_w1_pca_fact2_cluster_20)', 'log(ts_mean(add(total_market_capitalization, 0.0001), 20))', 'rank(log(add(abs(operating_revenue_other), 0.0001)))', 'scale(mdl109_matrix_cfoyld)', 'power(max(analyst_roe_revision_

 20%|██        | 4/20 [19:15<1:17:01, 288.84s/it]


Error on runtime (attempt 1/10): HTTPSConnectionPool(host='api.worldquantbrain.com', port=443): Max retries exceeded with url: /simulations/3SuNhs9FO4M5bUM125rHi88R (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x0000023B8943EAD0>: Failed to resolve 'api.worldquantbrain.com' ([Errno 11001] getaddrinfo failed)"))


2026-11-01 12:10:54,431 - ace - INFO - Trying re-login, wait 100 seconds


Starting iterative alpha generation
Error on runtime (attempt 2/10): HTTPSConnectionPool(host='api.worldquantbrain.com', port=443): Max retries exceeded with url: /operators (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x0000023B8943F4D0>: Failed to resolve 'api.worldquantbrain.com' ([Errno 11001] getaddrinfo failed)"))
Starting iterative alpha generation
Error on runtime (attempt 3/10): HTTPSConnectionPool(host='api.worldquantbrain.com', port=443): Max retries exceeded with url: /operators (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x0000023B8943F9D0>: Failed to resolve 'api.worldquantbrain.com' ([Errno 11001] getaddrinfo failed)"))
Starting iterative alpha generation
Error on runtime (attempt 4/10): HTTPSConnectionPool(host='api.worldquantbrain.com', port=443): Max retries exceeded with url: /operators (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x0000023B8943FD90>: Failed to resolve

2026-11-01 12:12:34,479 - ace - INFO - Trying re-login, wait 100 seconds


Starting iterative alpha generation
Error on runtime (attempt 6/10): HTTPSConnectionPool(host='api.worldquantbrain.com', port=443): Max retries exceeded with url: /operators (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x0000023B89F94550>: Failed to resolve 'api.worldquantbrain.com' ([Errno 11001] getaddrinfo failed)"))
Starting iterative alpha generation
Error on runtime (attempt 7/10): HTTPSConnectionPool(host='api.worldquantbrain.com', port=443): Max retries exceeded with url: /operators (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x0000023B89F95450>: Failed to resolve 'api.worldquantbrain.com' ([Errno 11001] getaddrinfo failed)"))
Starting iterative alpha generation
Error on runtime (attempt 8/10): HTTPSConnectionPool(host='api.worldquantbrain.com', port=443): Max retries exceeded with url: /operators (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x0000023B89F95810>: Failed to resolve

  0%|          | 0/20 [00:00<?, ?it/s]2026-11-01 12:15:37,535 - ace - ERROR - Child Simulation failed: {'id': '2xjAlJ7b94YjbKg1aHZZtke6', 'type': 'REGULAR', 'status': 'ERROR', 'message': 'Invalid number of inputs : 2, should be exactly 3 input(s). <linkToCommonErrorMessages>Learn more</linkToCommonErrorMessages>', 'location': {'line': 1, 'start': 0, 'end': 92, 'property': 'regular'}, 'links': {'linkToCommonErrorMessages': 'https://support.worldquantbrain.com/hc/en-us/articles/18423410021783--BRAIN-TIPS-Troubleshooting-common-error-messages'}}
2026-11-01 12:15:37,920 - ace - ERROR - Child Simulation failed: {'id': 'GliJM2MA4SEck516KRs4tKo', 'type': 'REGULAR', 'status': 'ERROR', 'message': 'Invalid number of inputs : 2, should be exactly 3 input(s). <linkToCommonErrorMessages>Learn more</linkToCommonErrorMessages>', 'location': {'line': 1, 'start': 0, 'end': 92, 'property': 'regular'}, 'links': {'linkToCommonErrorMessages': 'https://support.worldquantbrain.com/hc/en-us/articles/184234100

Error on runtime (attempt 1/10): ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
Starting iterative alpha generation
Iteration 1/2
Simulating expressions: ['ts_corr(rank(quarterly_retained_earnings), ts_delay(rank(total_market_capitalization), 5), 10)', 'normalize(subtract(mdl264_1l_nscf, ts_mean(mdl264_1l_nscf, 20)))', 'if_else(greater(reported_earnings_previous_quarter, 0), log(reported_earnings_previous_quarter), 0)', 'scale(ts_std_dev(forecasted_surprise_pct_ebitda_12m, 60))', 'rank(divide(analyst_ebitda_revision_ntm_3m, abs(analyst_roe_revision_ntm_3m)))', 'winsorize(ts_rank(market_implied_10yr_eps_growth_rate, 250))', 'ts_delta(log(reported_revenue_last_quarter), 10)', 'group_neutralize(rank(mdl264_vol_vo_p_12m_class), oth455_customer_n2v_p10_q50_w1_pca_fact2_cluster_20)', 'sqrt(abs(add(mdl264_2l_m3r_1yf_spd_se, mdl264_1l_nscf)))', 'log(ts_product(add(reported_earnings_previous_quarter, 0.0001), 5))', 'power(ts_delay(quarterly_icex_val

100%|██████████| 20/20 [00:10<00:00,  1.87it/s]


Complete biometrics authentication and press any key to continue: 
https://api.worldquantbrain.com/authentication/persona?inquiry=inq_AUAgjsUNxKxDyNidQaV5ZL7yvukzAx



In [ ]:
abs(float(s.iloc[0,5]))

NameError: name 's' is not defined